# Extract Unique Titles for LLM O*NET Mapping

Extract all unique `seniority_removed_title` values from Step 3 no_match rows.
These will be the input for LLM-based O*NET normalization (Phase 3).

**Source**: `step3_with_primary_tag` (parquet) — previously used `step3_csv`.

In [3]:
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path

STEP3_PARQUET = Path('../../data/processed/step3_with_primary_tag')
OUTPUT_DIR = Path('../../data/processed/step4_llm_onet_normalization/phase2_extract_unique_titles')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Read parquet via pyarrow directly (pandas read_parquet has Python 3.14 compatibility issue)
table = pq.read_table(STEP3_PARQUET)
df = table.to_pandas()

print(f'Total rows: {len(df):,}')
print(f'Columns: {list(df.columns)}')
print(f'\nMatch type distribution:')
print(df['match_type'].value_counts())

Total rows: 123,849
Columns: ['id', 'title', 'description', 'seniority', 'seniority_removed_title', 'primary_tag', 'match_type']

Match type distribution:
match_type
no_match      102322
partial        14452
exact_main      4979
exact_sub       2096
Name: count, dtype: int64


In [4]:
no_match = df[df['match_type'] == 'no_match']

unique_titles = no_match['seniority_removed_title'].drop_duplicates().sort_values().reset_index(drop=True)

print(f'no_match rows   : {len(no_match):,}')
print(f'unique titles   : {len(unique_titles):,}')
print(f'\nSample (first 10):')
for i, t in enumerate(unique_titles.head(10)):
    print(f'  {i+1}. {t}')

no_match rows   : 102,322
unique titles   : 58,980

Sample (first 10):
  1. 
  2. ![
  3. " Integration Engineer - IBM ACE/IIB Administrator/Developer"
  4. "Students in Cyber Security" National Research Study
  5. "Students in Cyber Security" Undergrad Interns
  6. # Residential Direct Support
  7. # Selling and Support Captain, Southdale Center - Full Time
  8. #13437 - Technical Salesforce Architect
  9. #13476 Manual Tester
  10. #13635 - Salesforce Manual Tester


In [5]:
import pyarrow as pa

output_path = OUTPUT_DIR / 'no_match_unique_titles.parquet'

df_output = unique_titles.to_frame(name='seniority_removed_title')
table_out = pa.Table.from_pandas(df_output)
pq.write_table(table_out, output_path)

# verify
verify = pq.read_table(output_path).to_pandas()
print(f'Saved: {output_path}')
print(f'Rows : {len(verify):,}')
print(f'Columns: {list(verify.columns)}')
print(f'\nHead:')
verify.head()

Saved: ..\..\data\processed\step4_llm_onet_normalization\phase2_extract_unique_titles\no_match_unique_titles.parquet
Rows : 58,980
Columns: ['seniority_removed_title']

Head:


,seniority_removed_title
0,
1,![
2,""" Integration Engineer - IBM ACE/IIB Administr..."
3,"""Students in Cyber Security"" National Research..."
4,"""Students in Cyber Security"" Undergrad Interns"
